In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_curve, auc, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
DATA_DIR = "data"

dataset_paths = {
    "dns_testing": f"{DATA_DIR}/DNS-testing.parquet",
    "ldap_testing": f"{DATA_DIR}/LDAP-testing.parquet",
    "ldap_training": f"{DATA_DIR}/LDAP-training.parquet",
    "mssql_testing": f"{DATA_DIR}/MSSQL-testing.parquet",
    "mssql_training": f"{DATA_DIR}/MSSQL-training.parquet",
    "ntp_testing": f"{DATA_DIR}/NTP-testing.parquet",
    "netbios_training": f"{DATA_DIR}/NetBIOS-training.parquet",
    "netbios_testing": f"{DATA_DIR}/NetBIOS-testing.parquet",
    "portmap_training": f"{DATA_DIR}/Portmap-training.parquet",
    "snmp_testing": f"{DATA_DIR}/SNMP-testing.parquet",
    "syn_testing": f"{DATA_DIR}/Syn-testing.parquet",
    "syn_training": f"{DATA_DIR}/Syn-training.parquet",
    "tftp_testing": f"{DATA_DIR}/TFTP-testing.parquet",
    "udp_testing": f"{DATA_DIR}/UDP-testing.parquet",
    "udp_training": f"{DATA_DIR}/UDP-training.parquet",
    "udplag_testing": f"{DATA_DIR}/UDPLag-testing.parquet",
    "udplag_training": f"{DATA_DIR}/UDPLag-training.parquet",
}

In [3]:
print("Loading datasets...")
dataframes = {name: pd.read_parquet(p) for name, p in dataset_paths.items()}
dataframes_list = list(dataframes.items())
print(f"Loaded {len(dataframes)} files")

Loading datasets...
Loaded 17 files


In [4]:
label_mapping = {
    'LDAP': 'DrDoS_LDAP',
    'MSSQL': 'DrDoS_MSSQL',
    'NetBIOS': 'DrDoS_NetBIOS',
    'UDP-lag': 'DDoS_UDP_Lag',
    'UDPLag': 'DDoS_UDP_Lag',
    'UDP': 'DrDoS_UDP',        # FIXED: was 'DDoS_UDP', caused duplicate class
    'Syn': 'DDoS_SYN',
    'Portmap': 'DrDoS_Portmap',
    'TFTP': 'DrDoS_TFTP',
    'WebDDoS': 'DDoS_Web'
}

for name, df in dataframes.items():
    df['Label'] = df['Label'].map(lambda x: label_mapping.get(x, x))

In [5]:
combined_datasets = {}
for name, df in dataframes_list:
    base_name = name.split('_')[0]
    if base_name not in combined_datasets:
        combined_datasets[base_name] = df
    else:
        combined_datasets[base_name] = pd.concat([combined_datasets[base_name], df], ignore_index=True)

df = pd.concat(combined_datasets.values(), ignore_index=True)
print(f"Combined shape: {df.shape}")

Combined shape: (431371, 78)


In [6]:
print(f"Number of unique labels: {df['Label'].nunique()}")
print(df['Label'].value_counts())

Number of unique labels: 13
Label
DrDoS_NTP        121368
DrDoS_TFTP        98917
Benign            97831
DDoS_SYN          49373
DrDoS_UDP         28510
DrDoS_MSSQL       14735
DDoS_UDP_Lag       8927
DrDoS_DNS          3669
DrDoS_LDAP         3346
DrDoS_SNMP         2717
DrDoS_NetBIOS      1242
DrDoS_Portmap       685
DDoS_Web             51
Name: count, dtype: int64


In [7]:
# Check for infinities and NaNs before cleaning
print("Infinite values per column (top 10):")
print(np.isinf(df.select_dtypes(include=[np.number])).sum().sort_values(ascending=False).head(10))

print(f"\nTotal NaNs: {df.isna().sum().sum()}")
print(f"Total duplicate rows: {df.duplicated().sum()}")

Infinite values per column (top 10):
Protocol                    0
Flow Duration               0
Total Fwd Packets           0
Total Backward Packets      0
Fwd Packets Length Total    0
Bwd Packets Length Total    0
Fwd Packet Length Max       0
Fwd Packet Length Min       0
Fwd Packet Length Mean      0
Fwd Packet Length Std       0
dtype: int64

Total NaNs: 0
Total duplicate rows: 9258


In [8]:
# Drop duplicates first — before splitting, so no duplicate row ends up in both train and test
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df.shape}")

# Identify non-numeric / identifier columns that shouldn't be used as model features
non_feature_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNon-numeric columns (excluded from features): {non_feature_cols}")

# Feature columns = everything numeric, excluding the label
feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumber of numeric feature columns: {len(feature_cols)}")
print(feature_cols)

Shape after removing duplicates: (422113, 78)

Non-numeric columns (excluded from features): ['Label']

Number of numeric feature columns: 77
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Cou

In [9]:
from sklearn.preprocessing import LabelEncoder

# Encode labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['Label'])

print("Label classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

X = df[feature_cols].values

# Split BEFORE scaling — this is the fix for the leakage issue
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded
)

print(f"\nTrain shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Label classes: ['Benign' 'DDoS_SYN' 'DDoS_UDP_Lag' 'DDoS_Web' 'DrDoS_DNS' 'DrDoS_LDAP'
 'DrDoS_MSSQL' 'DrDoS_NTP' 'DrDoS_NetBIOS' 'DrDoS_Portmap' 'DrDoS_SNMP'
 'DrDoS_TFTP' 'DrDoS_UDP']
Number of classes: 13

Train shape: (316584, 77)
Test shape: (105529, 77)


In [10]:
scaler = MinMaxScaler()

# Fit only on training data — test data must never influence scaler statistics
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the scaler fitted on train (no fitting here)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"X_train_scaled range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"X_test_scaled range: [{X_test_scaled.min():.4f}, {X_test_scaled.max():.4f}]")

Scaling complete.
X_train_scaled range: [0.0000, 1.0000]
X_test_scaled range: [0.0000, 4.5321]


In [11]:
benign_label_idx = list(label_encoder.classes_).index('Benign')
print(f"Benign label index: {benign_label_idx}")

# Autoencoder trains only on benign traffic from the TRAINING set
benign_mask_train = (y_train == benign_label_idx)
X_train_benign = X_train_scaled[benign_mask_train]

print(f"Benign training samples: {X_train_benign.shape[0]}")

# Reshape for LSTM: (samples, timesteps, features) — using 1 timestep since these are flow-level, not sequence, features
X_train_benign_lstm = X_train_benign.reshape((X_train_benign.shape[0], 1, X_train_benign.shape[1]))
X_test_scaled_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

print(f"LSTM input shape (train benign): {X_train_benign_lstm.shape}")
print(f"LSTM input shape (test, all classes): {X_test_scaled_lstm.shape}")

Benign label index: 0
Benign training samples: 71028
LSTM input shape (train benign): (71028, 1, 77)
LSTM input shape (test, all classes): (105529, 1, 77)


In [12]:
n_features = X_train_benign_lstm.shape[2]

# Encoder
input_layer = Input(shape=(1, n_features))
encoded = LSTM(64, activation='relu', return_sequences=True,
               kernel_regularizer=regularizers.l2(1e-4))(input_layer)
encoded = BatchNormalization()(encoded)
encoded = Dropout(0.2)(encoded)
encoded = LSTM(32, activation='relu', return_sequences=False)(encoded)

# Bottleneck -> Decoder
decoded = tf.keras.layers.RepeatVector(1)(encoded)
decoded = LSTM(32, activation='relu', return_sequences=True)(decoded)
decoded = BatchNormalization()(decoded)
decoded = Dropout(0.2)(decoded)
decoded = LSTM(64, activation='relu', return_sequences=True)(decoded)
decoded = tf.keras.layers.TimeDistributed(Dense(n_features))(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 77)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 1, 64)          │        36,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 1, 32)          │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 1, 64)          │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 1, 77)          │         5,005 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,309 (341.05 KB)

 Trainable params: 87,117 (340.30 KB)

 Non-trainable params: 192 (768.00 B)

In [13]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
checkpoint = ModelCheckpoint('models/autoencoder_model.keras', monitor='val_loss', save_best_only=True)

history = autoencoder.fit(
    X_train_benign_lstm, X_train_benign_lstm,
    epochs=50,
    batch_size=256,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

Epoch 1/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0157 - val_loss: 0.0339 - learning_rate: 0.0010
Epoch 2/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0025 - val_loss: 0.0164 - learning_rate: 0.0010
Epoch 3/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0017 - val_loss: 0.0018 - learning_rate: 0.0010
Epoch 4/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0012 - val_loss: 6.7152e-04 - learning_rate: 0.0010
Epoch 5/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 9.6357e-04 - val_loss: 5.6359e-04 - learning_rate: 0.0010
Epoch 6/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 8.2714e-04 - val_loss: 5.1361e-04 - learning_rate: 0.0010
Epoch 7/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.3580e-04 - val_loss: 5.5353e-04 - learning_rate: 0.0010
Epoch 8/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.7458e-04 - val_loss: 4.4324e-04 - learning_rate: 0.0010
Epoch 9/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 6.3301e-04 - val_loss: 4